# 2 ПЗ. Нарезка видео. Бокарев Никита М3О-312Б-23

Импорт библиотек

In [3]:
import os
import cv2
import yt_dlp
import requests
from tqdm import tqdm

Настройка параметров

In [11]:
VIDEO_SOURCE = "url"   # "url" или "local"

VIDEO_URL = "https://rutube.ru/shorts/6c48f15c9ad2a7a6250736b7375fe767"  # ссылка на видео
LOCAL_VIDEO_PATH = "video.mp4"               # путь к локальному видео

DOWNLOAD_QUALITY = "best"   # best / 720 / 480

OUTPUT_FOLDER = "frames"

FRAME_RATE = 2   # кадров в секунду

Загрузка видео (Rutube, YouTube)

In [5]:
def download_video(url, quality="best", output="video.mp4"):

    ydl_opts = {
        "outtmpl": output,
        "format": quality,
        "quiet": False
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    return output


Загрузка видеофайла (например, яндекс диск)

In [6]:
def download_file(url, filename):

    response = requests.get(url, stream=True)

    total = int(response.headers.get("content-length", 0))

    with open(filename, "wb") as file, tqdm(
        desc="Downloading",
        total=total,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:

        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

    return filename

Нарезка видео на кадры

In [7]:
def extract_frames(video_path, output_folder, frame_rate):

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)

    frame_interval = int(fps / frame_rate)

    count = 0
    saved = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if count % frame_interval == 0:

            filename = os.path.join(output_folder, f"frame_{saved:06d}.jpg")

            cv2.imwrite(filename, frame)

            saved += 1

        count += 1

    cap.release()

    print("Saved frames:", saved)

Запуск

In [12]:
video_path = None

if VIDEO_SOURCE == "url":

    print("Downloading video...")

    video_path = download_video(
        VIDEO_URL,
        quality=DOWNLOAD_QUALITY,
        output="downloaded_video.mp4"
    )

elif VIDEO_SOURCE == "local":

    video_path = LOCAL_VIDEO_PATH

else:
    raise ValueError("VIDEO_SOURCE должен быть 'url' или 'local'")


print("Extracting frames...")

extract_frames(
    video_path,
    OUTPUT_FOLDER,
    FRAME_RATE
)


[generic] Extracting URL: https://rutube.ru/shorts/6c48f15c9ad2a7a6250736b7375fe767
[generic] 6c48f15c9ad2a7a6250736b7375fe767: Downloading webpage
[redirect] Following redirect to https://rutube.ru/shorts/6c48f15c9ad2a7a6250736b7375fe767/
[generic] Extracting URL: https://rutube.ru/shorts/6c48f15c9ad2a7a6250736b7375fe767/
[generic] 6c48f15c9ad2a7a6250736b7375fe767: Downloading webpage


[generic] 6c48f15c9ad2a7a6250736b7375fe767: Extracting information
[rutube] Extracting URL: https://rutube.ru/play/embed/6c48f15c9ad2a7a6250736b7375fe767?isfulltab=true
[rutube] 6c48f15c9ad2a7a6250736b7375fe767: Downloading video JSON
[rutube] 6c48f15c9ad2a7a6250736b7375fe767: Downloading options JSON
[rutube] 6c48f15c9ad2a7a6250736b7375fe767: Downloading m3u8 information
[rutube] 6c48f15c9ad2a7a6250736b7375fe767: Downloading m3u8 information
[info] 6c48f15c9ad2a7a6250736b7375fe767: Downloading 1 format(s): m3u8-2927-1
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 14
[download] Destination: downloaded_video.mp4
[download] 100% of   17.55MiB in 00:00:06 at 2.63MiB/s                  


Extracting frames...
Saved frames: 97
